In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

### GloFAS — Global Flood Awareness System (Copernicus EWDS)

**GloFAS** provides daily river discharge estimates worldwide from the **LISFLOOD** hydrological
model forced with **ERA5 reanalysis**. Part of the Copernicus Emergency Management Service.

#### ❓ What is `dis24`?

`dis24` is the **mean daily river discharge** (m³/s) — the average volumetric flow rate over 24 hours.
It is a **rate**, not a volume. To convert to daily volume:
```
Q (m³/day) = dis24 (m³/s) × 86400 s/day
```

#### 🔧 What is LISFLOOD?

LISFLOOD is a distributed hydrological model developed by the JRC that simulates snow, soil moisture,
groundwater, and river routing on a 0.05° grid. It is **not calibrated to local observations** —
it uses global parameter sets from soil maps and land cover data. This means:
- ✅ Seasonal patterns (flood timing, low-flow periods) generally well captured
- ⚠️ Magnitudes may differ from observations by **20–50%** without local correction
- ❌ Not recommended for flood frequency analysis without bias correction

#### 🆚 GloFAS vs observed discharge

| Use case | GloFAS | Observations (GRDC/USGS/SAIH) |
|----------|--------|------------------------------|
| Ungauged catchments | ✅ Only option | ❌ Not available |
| Long historical record (1979–) | ✅ Consistent | ⚠️ Gaps common |
| Calibration reference | ❌ Not recommended | ✅ Ground truth |
| Small catchments (<500 km²) | ⚠️ 5 km grid is coarse | ✅ Better |
| Flood frequency analysis | ⚠️ Bias correction needed | ✅ Preferred |

#### 📐 Spatial resolution note

The 0.05° grid (~5 km) means **river channels narrower than ~5 km are not explicitly resolved**.
For catchments smaller than ~500 km², the extracted grid cell may not align with the actual channel.
Always verify by overlaying the GloFAS point on a DEM-derived river network.

#### 💾 Data volume (Iberian Peninsula)

- Area: 183 lat × 302 lon = ~55k cells
- 1 year of daily `dis24`: **~80 MB**
- 40 years (1979–2019): **~3.2 GB** — use year-by-year download to avoid CDS queue timeouts

#### 💡 Key characteristics

| Property | Value |
|----------|-------|
| Variable | `dis24` — mean daily discharge (m³/s) |
| Spatial resolution | 0.05° (~5 km) |
| Temporal coverage | 1979 – present |
| Model | LISFLOOD |
| Forcing | ERA5 reanalysis |
| Current version | v4.0 (improved calibration vs v3.1) |

#### ⚙️ Prerequisites

1. Register at https://ewds.climate.copernicus.eu (different from standard CDS account)
2. Accept terms for `cems-glofas-historical` on the portal
3. Configure `~/.cdsapirc`:
   ```ini
   url: https://ewds.climate.copernicus.eu/api
   key: <your-personal-api-key>
   ```
4. `pip install cdsapi xarray netCDF4`

#### 🔗 Links
- Portal: https://ewds.climate.copernicus.eu
- Dataset: https://ewds.climate.copernicus.eu/datasets/cems-glofas-historical


In [ ]:
from pyhydra.data_sources.river_discharge import (
    download_glofas,
    download_glofas_by_year,
    read_glofas_nc,
)
from pyhydra.utils import interactive_map
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path

# Portable repo-root / data-dir resolution (local clone, Docker, Azure ML)
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))


## 🗺️ Select area of interest

Draw a **rectangle** on the map to define the download region. Read `coordinates_list[0]` in the next cell.

In [ ]:
coordinates_list = interactive_map(zoom=4, center=(40, -5))

In [ ]:
# === CONFIGURATION ===
API_URL = 'https://ewds.climate.copernicus.eu/api'
API_KEY = 'YOUR_EWDS_API_KEY'   # or None to use ~/.cdsapirc
# Get your key at https://ewds.climate.copernicus.eu/profile

AREA = coordinates_list[0] if coordinates_list else [44, -10, 35, 5]  # [N, W, S, E]
YEARS = list(range(2000, 2021))
OUTPUT_DIR = str(DATA_DIR / 'glofas')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SYSTEM_VERSION = 'version_4_0'
PRODUCT_TYPE   = 'consolidated'
FILE_FORMAT    = 'netcdf4'
MONTHS         = list(range(1, 13))

LAT = 39.5
LON = -3.5

print(f'Area (N, W, S, E): {AREA}')
print(f'Output directory : {OUTPUT_DIR}')


---
## 1. ⬇️ Download — full range (single request)

Submits one CDS request covering all years at once. Suitable for short periods; may time out for large date ranges.

In [ ]:
if API_KEY == 'YOUR_EWDS_API_KEY':
    print('[GloFAS] API key not configured — set API_KEY above.')
    print('[GloFAS] Get your key at: https://ewds.climate.copernicus.eu/profile')
    paths = []
else:
    paths = download_glofas(
        area=AREA,
        years=YEARS,
        output_dir=OUTPUT_DIR,
        system_version=SYSTEM_VERSION,
        product_type=PRODUCT_TYPE,
        months=MONTHS,
        file_format=FILE_FORMAT,
        api_key=API_KEY,
        api_url=API_URL,
    )
    print('Downloaded:', paths)

[GloFAS] API key not configured — set API_KEY above.
[GloFAS] Get your key at: https://ewds.climate.copernicus.eu/profile


---
## 2. ⬇️ Download — year by year (recommended)

Submits one CDS request per year. Avoids timeout issues with large requests and allows partial downloads to resume.

In [ ]:
if API_KEY == 'YOUR_EWDS_API_KEY':
    print('[GloFAS] API key not configured — skipping by-year download.')
    paths = []
else:
    paths = download_glofas_by_year(
        area=AREA,
        years=YEARS,
        output_dir=OUTPUT_DIR,
        system_version=SYSTEM_VERSION,
        product_type=PRODUCT_TYPE,
        months=MONTHS,
        file_format=FILE_FORMAT,
        api_key=API_KEY,
        api_url=API_URL,
    )
    print(f'Downloaded {len(paths)} files:')
    for p in paths:
        print(' ', p)

[GloFAS] API key not configured — skipping by-year download.


---
## 3. 📖 Reading downloaded NetCDF files

`read_glofas_nc(file, lat, lon)` extracts the `dis24` time series at the nearest grid cell
and returns a DataFrame with columns `['date', 'discharge']`.

**Coordinate alignment:** The function uses `method='nearest'` — the extracted cell may be
2–3 km from the target point. For a small catchment outlet, verify the cell location:
```python
import xarray as xr
ds = xr.open_dataset('glofas_2000.nc')
# Check which cell was selected
ds.sel(latitude=LAT, longitude=LON, method='nearest').latitude.values
```

For multi-year datasets, concatenate all annual files:
```python
from pathlib import Path
dfs = [read_glofas_nc(f, lat=LAT, lon=LON) for f in sorted(Path(OUTPUT_DIR).glob('*.nc'))]
df_all = pd.concat(dfs).set_index('date').sort_index()
```


In [ ]:
try:
    # Extract discharge at a specific coordinate from a single .nc file
    NC_FILE = f'{OUTPUT_DIR}/glofas_cems-glofas-historical_2000.nc'
    
    df_glofas = read_glofas_nc(NC_FILE, lat=LAT, lon=LON)
    df_glofas = df_glofas.set_index('date')
    print(df_glofas.describe())
    df_glofas.head()
except FileNotFoundError as e:
    print(f'[GloFAS] No downloaded files found — run download cells first. ({e})')
    df_glofas = None

[GloFAS] No downloaded files found — run download cells first. ([Errno 2] No such file or directory: '/workspace/data/glofas/glofas_cems-glofas-historical_2000.nc')


In [ ]:
# Load and concatenate multiple yearly files into a single series
dfs = []
for year in YEARS:
    nc = f'{OUTPUT_DIR}/glofas_cems-glofas-historical_{year}.nc'
    if Path(nc).exists():
        dfs.append(read_glofas_nc(nc, lat=LAT, lon=LON))

if not dfs:
    print('[GloFAS] No .nc files found in output directory — download first.')
    df_all = None
else:
    df_all = pd.concat(dfs).set_index('date').sort_index()
if df_all is not None:
    print(f'Period : {df_all.index[0].date()} → {df_all.index[-1].date()}')
    print(f'Mean Q : {df_all["discharge"].mean():.2f} m³/s')
    print(f'Max  Q : {df_all["discharge"].max():.1f} m³/s')
    display(df_all.head())

[GloFAS] No .nc files found in output directory — download first.


---
## 4. 📊 Visualisation and interpretation

The plots show:
1. **Daily hydrograph** — peaks correspond to flood events; flat periods = low flow / summer minimum
2. **Seasonal regime** — identifies the hydrological regime:
   - **Nival**: peak in spring (snowmelt, e.g., Ebro headwaters)
   - **Pluvial**: peak in winter/autumn (rain-dominated, e.g., Guadalquivir)
   - **Mixed**: double peak (snow + rain, typical of large Iberian basins)

> Note that GloFAS seasonal patterns are generally more reliable than absolute magnitudes.


In [ ]:
if df_all is None:
    print('[GloFAS] No data loaded — skipping plots.')
else:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7))
    
    axes[0].plot(df_all.index, df_all['discharge'], lw=0.6, color='darkorange')
    axes[0].set_ylabel('GloFAS discharge (m³/s)')
    axes[0].set_title(f'GloFAS v4.0 · ({LAT}°N, {LON}°E)', fontsize=13)
    axes[0].grid(alpha=0.3)
    
    monthly = df_all['discharge'].groupby(df_all.index.month).mean()
    axes[1].bar(monthly.index, monthly.values, color='darkorange', alpha=0.85)
    axes[1].set_xticks(range(1, 13))
    axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                              'Jul','Aug','Sep','Oct','Nov','Dec'])
    axes[1].set_ylabel('Mean discharge (m³/s)')
    axes[1].set_title('Seasonal regime')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

[GloFAS] No data loaded — skipping plots.


---
## ✅ Validation against observations

Before using GloFAS for flood frequency analysis, validate against gauged observations:

```python
from pyhydra.climate import nash_sutcliffe

# Align periods
common = df_obs.index.intersection(df_glofas.index)
obs = df_obs.loc[common, 'Q']
sim = df_glofas.loc[common, 'discharge']

# Metrics
bias   = sim.mean() / obs.mean()      # multiplicative bias
nse    = nash_sutcliffe(obs, sim)      # Nash-Sutcliffe Efficiency
kge    = kling_gupta(obs, sim)         # Kling-Gupta Efficiency
print(f'Bias: {bias:.2f}  NSE: {nse:.2f}  KGE: {kge:.2f}')
```

| Metric | Acceptable | Good |
|--------|-----------|------|
| Nash-Sutcliffe (NSE) | > 0.5 | > 0.7 |
| KGE | > 0.5 | > 0.75 |
| Bias | 0.8 – 1.2 | 0.9 – 1.1 |

If NSE < 0, GloFAS is **worse than using the mean** — apply multiplicative bias correction at minimum.

#### ⚠️ Flood frequency caution

ERA5 underestimates extreme rainfall intensities, which propagates to LISFLOOD.
GloFAS typically **underestimates high return-period flows** (T > 50 years).
Correct with a quantile mapping approach trained on the observed period before use in design.
